# Lab 8 — Automation, Headless & CI (Features #23–#27)
**AmbariCloud FDE Workshop · Day 2 bridge** · This is the foundation of the QA pipeline deck.

## Feature #23 — Headless mode: `claude -p`
`-p` / `--print`: run ONE prompt non-interactively, print, exit. For scripts and pipelines.
- `--output-format json` (or `stream-json`) so a program can parse the result
- reads stdin: `cat log | claude -p 'explain'`
- guard cost: `--max-budget-usd`, `--max-turns`
**Watch for:** in `-p` mode nobody answers permission prompts — pair with `--allowedTools` or it stops on the first write.


### Try it (runnable)

In [ ]:
%%bash
cd ~/workshop-todo 2>/dev/null || cd ~
claude -p "list every file in this directory, one per line" --allowedTools Bash 2>&1 | tail -8

In [ ]:
%%bash
claude -p "summarise what a todo CLI app should do" --output-format json 2>/dev/null | python3 -c "
import json,sys
d=json.load(sys.stdin)
print('result:', d['result'][:200])
print('cost_usd:', d.get('total_cost_usd'))
print('session:', d.get('session_id'))" 

In [ ]:
%%bash
# Claude as a pipeline stage: pipe stderr of a failing run into it
echo "E   AssertionError: expected 3 items, got 2" | claude -p "one line: why did this test fail?" 2>&1 | tail -2

## Feature #24 — `--dangerously-skip-permissions`: only in a sandbox
Removes the per-action approval loop — every tool call runs unasked (aka YOLO mode).

> ⚠️ **SECURITY** — with this flag Claude can run `rm -rf`, push to main, or exfiltrate files if a prompt injection says so. Use **only** inside a throwaway Docker container with no credentials and no network it does not need.

```bash
# inside a throwaway container ONLY
docker run --rm -it -v $PWD:/w -w /w node:20 bash
  npm i -g @anthropic-ai/claude-code
  claude --dangerously-skip-permissions -p "refactor todo.py into a package, keep tests green"
```
**Safer default for demos:** `claude --permission-mode acceptEdits`

## Feature #25 — `--allowedTools` / `--add-dir`: pre-authorise
- `--allowedTools "Bash(pytest *)" "Bash(git log *)" Edit` — pre-approve, with prefix patterns
- `--disallowedTools` — hard deny
- `--add-dir ../shared-lib` (or `/add-dir` in session) — grant extra directories
- `settings.json` `permissions.allow / deny` — same rules persist; commit to share with team
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
claude --allowedTools "Bash(pytest *)" Edit
> run the tests          # no prompt
> git push               # still asks (not allowed) — the boundary is visible
> /permissions           # view + add rules; writes .claude/settings.local.json
```
**Watch for:** patterns are prefix matches — `"Bash(git *)"` allows `git push --force` too. Allow the narrowest thing that unblocks you.


## Feature #26 — Git worktrees: a one-person team
A worktree = a branch checked out in its OWN directory. Several Claude instances edit one repo in parallel, no conflicts.
```bash
# terminal 1
claude -w colour   "add coloured output; commit"
# terminal 2
claude -w export   "add an export-csv command; commit"

git worktree list
git log --oneline --all --graph | head
# later: merge branches or open PRs
```
**Watch for:** both instances share `.git` but not working files — a `pip install` in one is not in the other; each worktree needs its own venv.

## Feature #27 — Claude in CI / GitHub: reviews and gates
Official GitHub Action `anthropics/claude-code-action` — `@claude` in issues/PRs, automatic PR review. Install from a session: `/install-github-app`.

`.github/workflows/claude-review.yml`:
```yaml
on: pull_request
jobs:
  review:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: anthropics/claude-code-action@v1
        with:
          anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
          prompt: "Review this PR for bugs and test gaps"
```
**Rules for unattended CI:** pin the action version, cap spend (`--max-budget-usd`), narrowest `--allowedTools`, `ANTHROPIC_API_KEY` in Actions secrets — never in the workflow file.


### Budget-guarded headless run (runnable)

In [ ]:
%%bash
claude -p "In exactly 5 bullet points: what makes a CI code-review bot safe?" --max-turns 1 2>&1 | tail -8